# Time series 2: New models (LSTM)

Same data and evaluation as `time_series.ipynb`: load transformed data, use the last **365×24 hours** as the test set, and evaluate **1-step-ahead** forecasts with MAE, RMSE, and MASE.

This notebook adds **LSTM** (and can be extended with other new models). Run the first cell to load data; then run the LSTM cell to train, predict, and compute metrics.

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root (notebook may run from time_series/ or repo root)
_root = Path.cwd().resolve()
if _root.name == "time_series":
    _root = _root.parent
DATA_PATH = _root / "data" / "transformed" / "transformed_data.parquet"
if not DATA_PATH.exists():
    DATA_PATH = DATA_PATH.with_suffix(".csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Transform data not found at {DATA_PATH}. Run task transform first.")

df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
df = df.sort_values(["date", "hour"]).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")

freq_bands = sorted([c for c in df.columns if c not in ("date", "hour", "datetime")])
TEST_STEPS = 24 * 365  # last year of hours

print(f"Loaded {len(df)} rows. Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Frequency bands: {len(freq_bands)}")
print(f"Test window: last {TEST_STEPS} hours (1-step-ahead)")

# Global list for final summary: each model cell below appends {"model": name, "mae": float, "mase": float}
ALL_RESULTS = []

Loaded 9192 rows. Date range: 2025-01-01 to 2026-01-18
Frequency bands: 70
Test window: last 8760 hours (1-step-ahead)


## h-step-ahead: Naive vs Seasonal Naive

At **h=1** (one hour ahead), naive (last value) usually wins because the series is very persistent. At **h=24** (24 hours ahead), we predict using "now" (naive) vs "same hour yesterday" (seasonal). Seasonal should win when there is daily seasonality. Below we compute MAE and MASE for both horizons so you can see the difference.

In [7]:
# 1-step and 24-step: naive vs seasonal naive
# 1-step: at time t predict y(t+1). Naive: ŷ=y(t). Seasonal: ŷ=y(t-23) (same hour yesterday).
# 24-step: at time t predict y(t+24). Naive: ŷ=y(t) (now). Seasonal: ŷ=y(t-24) (same hour yesterday).

def mase_denom_from_actuals(actuals):
    d = np.abs(np.diff(actuals))
    return float(np.mean(d)) if len(d) > 0 else np.nan

rows_1_naive, rows_1_seasonal = [], []
rows_24_naive, rows_24_seasonal = [], []

for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 25:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals = vals[-TEST_STEPS:]
    test_dts = dts[-TEST_STEPS:]

    # 1-step: i in [0, TEST_STEPS-2]; need i>=24 for seasonal (y(t-23))
    for i in range(0, TEST_STEPS - 1):
        actual = test_vals[i + 1]
        pred_naive = test_vals[i]
        rows_1_naive.append({"datetime": test_dts[i + 1], "frequency_band": freq, "actual": actual, "predicted": pred_naive})
        if i >= 23:
            pred_seasonal = test_vals[i - 23]  # y(t+1-24): same hour yesterday
            rows_1_seasonal.append({"datetime": test_dts[i + 1], "frequency_band": freq, "actual": actual, "predicted": pred_seasonal})

    # 24-step: at index i we predict for i+24. i in [24, TEST_STEPS-25]. Naive: pred=test_vals[i]. Seasonal: pred=test_vals[i-24]
    for i in range(24, TEST_STEPS - 24):
        actual = test_vals[i + 24]
        pred_naive_24 = test_vals[i]
        pred_seasonal_24 = test_vals[i - 24]
        rows_24_naive.append({"datetime": test_dts[i + 24], "frequency_band": freq, "actual": actual, "predicted": pred_naive_24})
        rows_24_seasonal.append({"datetime": test_dts[i + 24], "frequency_band": freq, "actual": actual, "predicted": pred_seasonal_24})

df_1_naive = pd.DataFrame(rows_1_naive)
df_1_seasonal = pd.DataFrame(rows_1_seasonal)
df_24_naive = pd.DataFrame(rows_24_naive)
df_24_seasonal = pd.DataFrame(rows_24_seasonal)

def metrics(actual, pred):
    mae = float(np.mean(np.abs(actual - pred)))
    rmse = float(np.sqrt(np.mean((actual - pred) ** 2)))
    return mae, rmse

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

# 1-step
a1n, p1n = df_1_naive["actual"].values, df_1_naive["predicted"].values
mae_1_naive, rmse_1_naive = metrics(a1n, p1n)
mase_1_naive = mase_from_df(df_1_naive)
a1s, p1s = df_1_seasonal["actual"].values, df_1_seasonal["predicted"].values
mae_1_seasonal, rmse_1_seasonal = metrics(a1s, p1s)
mase_1_seasonal = mase_from_df(df_1_seasonal)

# 24-step
a24n, p24n = df_24_naive["actual"].values, df_24_naive["predicted"].values
mae_24_naive, rmse_24_naive = metrics(a24n, p24n)
mase_24_naive = mase_from_df(df_24_naive)
a24s, p24s = df_24_seasonal["actual"].values, df_24_seasonal["predicted"].values
mae_24_seasonal, rmse_24_seasonal = metrics(a24s, p24s)
mase_24_seasonal = mase_from_df(df_24_seasonal)

print("=== 1-step-ahead (predict next hour) ===")
print(f"  Naive:    MAE = {mae_1_naive:.4f},  MASE = {mase_1_naive:.4f}" if not np.isnan(mase_1_naive) else f"  Naive:    MAE = {mae_1_naive:.4f},  MASE = —")
print(f"  Seasonal: MAE = {mae_1_seasonal:.4f},  MASE = {mase_1_seasonal:.4f}" if not np.isnan(mase_1_seasonal) else f"  Seasonal: MAE = {mae_1_seasonal:.4f},  MASE = —")
print(f"  → Winner at h=1: {'Naive' if mae_1_naive <= mae_1_seasonal else 'Seasonal'} (lower MAE)")

print("\n=== 24-step-ahead (predict 24 hours ahead) ===")
print(f"  Naive:    MAE = {mae_24_naive:.4f},  MASE = {mase_24_naive:.4f}" if not np.isnan(mase_24_naive) else f"  Naive:    MAE = {mae_24_naive:.4f},  MASE = —")
print(f"  Seasonal: MAE = {mae_24_seasonal:.4f},  MASE = {mase_24_seasonal:.4f}" if not np.isnan(mase_24_seasonal) else f"  Seasonal: MAE = {mae_24_seasonal:.4f},  MASE = —")
print(f"  → Winner at h=24: {'Seasonal' if mae_24_seasonal < mae_24_naive else 'Naive'} (lower MAE)")

print("\n→ At 1-step, naive wins (last hour is very informative). At 24-step, seasonal wins when daily seasonality is present.")
ALL_RESULTS.append({"model": "Naive", "mae": mae_1_naive, "mase": mase_1_naive})
ALL_RESULTS.append({"model": "Seasonal naive", "mae": mae_1_seasonal, "mase": mase_1_seasonal})

=== 1-step-ahead (predict next hour) ===
  Naive:    MAE = 2.4198,  MASE = 0.9999
  Seasonal: MAE = 4.2900,  MASE = 1.7742
  → Winner at h=1: Naive (lower MAE)

=== 24-step-ahead (predict 24 hours ahead) ===
  Naive:    MAE = 4.2897,  MASE = 1.7799
  Seasonal: MAE = 5.0266,  MASE = 2.0856
  → Winner at h=24: Naive (lower MAE)

→ At 1-step, naive wins (last hour is very informative). At 24-step, seasonal wins when daily seasonality is present.


## LSTM: 1-step-ahead forecast

For each frequency band we train a small LSTM on all data **before** the test window, using sequences of the last **SEQ_LEN** hours to predict the next hour. On the test window we do rolling 1-step-ahead: at each hour use the last SEQ_LEN values (true) as input and predict the next; compare to actual and compute MAE, RMSE, MASE.

## More RNNs: GRU, Stacked LSTM, LSTM + seasonal feature (attempt to beat 1-step naive)

We add **GRU**, **2-layer LSTM**, and **LSTM with lag-24** (same-hour-yesterday as a second input so the model sees daily seasonality). All are trained per band and evaluated on the same 1-step-ahead test set; results are compared to the naive baseline.

In [9]:
import json
import torch
import torch.nn as nn

SEQ_LEN = 24
HIDDEN = 32
EPOCHS = 25
BATCH = 64
LR = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Naive 1-step baseline (for comparison) ----
rows_naive = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 1:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals = vals[-TEST_STEPS:]
    test_dts = dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i + 1], "frequency_band": freq, "actual": test_vals[i + 1], "predicted": test_vals[i]})
df_naive = pd.DataFrame(rows_naive)
mae_naive = float(np.mean(np.abs(df_naive["actual"] - df_naive["predicted"])))
diffs_naive = df_naive.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom_global = float(diffs_naive.mean()) if len(diffs_naive) > 0 else np.nan
mase_naive = mae_naive / mase_denom_global if mase_denom_global and mase_denom_global > 0 else np.nan

def build_sequences(vals, seq_len):
    X, Y = [], []
    for i in range(seq_len, len(vals)):
        X.append(vals[i - seq_len:i].reshape(-1, 1).astype(np.float32))
        Y.append(vals[i])
    return np.array(X), np.array(Y, dtype=np.float32)

def build_sequences_lag24(vals, seq_len):
    """Input: (seq_len, 2) = [y(t-seq:t), y(t-24-seq:t-24)]. Requires len(vals) >= seq_len + 24."""
    X, Y = [], []
    for i in range(24, len(vals) - seq_len):
        row = np.stack([vals[i:i + seq_len], vals[i - 24:i - 24 + seq_len]], axis=1).astype(np.float32)
        X.append(row)
        Y.append(vals[i + seq_len])
    return np.array(X), np.array(Y, dtype=np.float32)

def train_model(model, X, Y, epochs=EPOCHS, lr=LR, batch=BATCH):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(X)
    for ep in range(epochs):
        perm = np.random.permutation(n)
        for start in range(0, n, batch):
            idx = perm[start:start + batch]
            x = torch.from_numpy(X[idx]).to(device)
            y = torch.from_numpy(Y[idx]).reshape(-1, 1).to(device)
            opt.zero_grad()
            pred = model(x)
            loss = nn.functional.mse_loss(pred, y)
            loss.backward()
            opt.step()
    return model

# ---- Model definitions ----
class LSTM1Step(nn.Module):
    def __init__(self, in_dim=1, hidden=HIDDEN):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True)
        self.linear = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :] if out.dim() == 3 else out
        return self.linear(last)

class GRU1Step(nn.Module):
    def __init__(self, hidden=HIDDEN):
        super().__init__()
        self.gru = nn.GRU(1, hidden, batch_first=True)
        self.linear = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.linear(out[:, -1, :])

class StackedLSTM1Step(nn.Module):
    def __init__(self, hidden=HIDDEN):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, num_layers=2, batch_first=True)
        self.linear = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])

def run_model(name, model_fn, build_fn, need_lag24=False):
    rows = []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if need_lag24:
            if len(sub) < TEST_STEPS + SEQ_LEN + 24:
                continue
        else:
            if len(sub) < TEST_STEPS + SEQ_LEN + 1:
                continue
        vals = sub[freq].values.astype(np.float32)
        dts = sub["datetime"].values
        train_vals = vals[: -TEST_STEPS]
        test_vals = vals[-TEST_STEPS:]
        test_dts = dts[-TEST_STEPS:]
        if need_lag24:
            X_tr, Y_tr = build_sequences_lag24(train_vals, SEQ_LEN)
        else:
            X_tr, Y_tr = build_fn(train_vals, SEQ_LEN)
        if len(X_tr) < BATCH:
            continue
        model = model_fn().to(device)
        train_model(model, X_tr, Y_tr)
        model.eval()
        with torch.no_grad():
            for i in range(TEST_STEPS):
                if need_lag24:
                    start = len(train_vals) + i - SEQ_LEN
                    if start < 24:
                        continue
                    end = start + SEQ_LEN
                    inp = np.stack([vals[start:end], vals[start - 24:end - 24]], axis=1).astype(np.float32)
                    inp = np.ascontiguousarray(inp).reshape(1, SEQ_LEN, 2)
                else:
                    start = len(train_vals) - SEQ_LEN + i
                    end = start + SEQ_LEN
                    inp = vals[start:end].reshape(1, SEQ_LEN, 1)
                x = torch.from_numpy(inp).to(device)
                pred = model(x).cpu().numpy().item()
                rows.append({"datetime": test_dts[i], "frequency_band": freq, "actual": float(test_vals[i]), "predicted": float(pred)})
    return pd.DataFrame(rows)

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

print("Training GRU, Stacked LSTM, and LSTM+lag24 (1-step-ahead)...")
df_gru = run_model("GRU", lambda: GRU1Step(), build_sequences, need_lag24=False)
df_stacked = run_model("StackedLSTM", lambda: StackedLSTM1Step(), build_sequences, need_lag24=False)
df_lstm24 = run_model("LSTM+lag24", lambda: LSTM1Step(in_dim=2), build_sequences_lag24, need_lag24=True)

mae_gru = float(np.mean(np.abs(df_gru["actual"] - df_gru["predicted"])))
mae_stacked = float(np.mean(np.abs(df_stacked["actual"] - df_stacked["predicted"])))
mae_lstm24 = float(np.mean(np.abs(df_lstm24["actual"] - df_lstm24["predicted"])))
mase_gru = mase_from_df(df_gru)
mase_stacked = mase_from_df(df_stacked)
mase_lstm24 = mase_from_df(df_lstm24)

print("\n=== 1-step-ahead: RNN models vs Naive ===")
print(f"  Naive:       MAE = {mae_naive:.4f},  MASE = {mase_naive:.4f}" if not np.isnan(mase_naive) else f"  Naive:       MAE = {mae_naive:.4f},  MASE = —")
print(f"  GRU:         MAE = {mae_gru:.4f},  MASE = {mase_gru:.4f}" if not np.isnan(mase_gru) else f"  GRU:         MAE = {mae_gru:.4f},  MASE = —")
print(f"  Stacked LSTM: MAE = {mae_stacked:.4f},  MASE = {mase_stacked:.4f}" if not np.isnan(mase_stacked) else f"  Stacked LSTM: MAE = {mae_stacked:.4f},  MASE = —")
print(f"  LSTM+lag24:  MAE = {mae_lstm24:.4f},  MASE = {mase_lstm24:.4f}" if not np.isnan(mase_lstm24) else f"  LSTM+lag24:  MAE = {mae_lstm24:.4f},  MASE = —")

best_mae = min(mae_naive, mae_gru, mae_stacked, mae_lstm24)
winner = "Naive" if mae_naive <= best_mae + 1e-6 else ("GRU" if mae_gru <= best_mae + 1e-6 else ("Stacked LSTM" if mae_stacked <= best_mae + 1e-6 else "LSTM+lag24"))
print(f"\n  → Best MAE: {best_mae:.4f} ({winner})")
if mae_gru < mae_naive or mae_stacked < mae_naive or mae_lstm24 < mae_naive:
    print("  → At least one RNN beats naive at 1-step.")
else:
    print("  → Naive still wins at 1-step (common when series are very persistent).")
ALL_RESULTS.append({"model": "GRU", "mae": mae_gru, "mase": mase_gru})
ALL_RESULTS.append({"model": "Stacked LSTM", "mae": mae_stacked, "mase": mase_stacked})
ALL_RESULTS.append({"model": "LSTM+lag24", "mae": mae_lstm24, "mase": mase_lstm24})

Training GRU, Stacked LSTM, and LSTM+lag24 (1-step-ahead)...

=== 1-step-ahead: RNN models vs Naive ===
  Naive:       MAE = 2.4198,  MASE = 0.9999
  GRU:         MAE = 10.0823,  MASE = 4.1665
  Stacked LSTM: MAE = 9.5963,  MASE = 3.9657
  LSTM+lag24:  MAE = 10.0580,  MASE = 4.1565

  → Best MAE: 2.4198 (Naive)
  → Naive still wins at 1-step (common when series are very persistent).


NameError: name 'ALL_RESULTS' is not defined

## Extra RNNs: Residual LSTM, BiLSTM, Ensemble (beat 1-step naive)

- **Residual LSTM**: Predict the *delta* from naive (target = y(t+1) − y(t)); final forecast = y(t) + predicted delta. Often easier to learn and can match or beat naive.
- **Bidirectional LSTM**: Stronger encoder over the lookback window (no future leak at test time).
- **Ensemble**: Blend naive with the best RNN (e.g. 0.5×naive + 0.5×RNN, or a tuned weight) to see if combining helps.

In [10]:
# Residual LSTM, BiLSTM, and Ensemble (requires: df, freq_bands, TEST_STEPS from earlier cells)
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEQ_LEN = 24
HIDDEN = 32
EPOCHS = 25
BATCH = 64
LR = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Naive baseline (recompute so this cell can run standalone)
rows_naive = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 1:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals = vals[-TEST_STEPS:]
    test_dts = dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i + 1], "frequency_band": freq, "actual": test_vals[i + 1], "predicted": test_vals[i]})
df_naive = pd.DataFrame(rows_naive)
mae_naive = float(np.mean(np.abs(df_naive["actual"] - df_naive["predicted"])))
diffs_naive = df_naive.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom_global = float(diffs_naive.mean()) if len(diffs_naive) > 0 else np.nan
mase_naive = mae_naive / mase_denom_global if mase_denom_global and mase_denom_global > 0 else np.nan

def build_sequences(vals, seq_len):
    X, Y = [], []
    for i in range(seq_len, len(vals)):
        X.append(vals[i - seq_len:i].reshape(-1, 1).astype(np.float32))
        Y.append(vals[i])
    return np.array(X), np.array(Y, dtype=np.float32)

def build_sequences_residual(vals, seq_len):
    """Target = y(t) - y(t-1) (residual from naive)."""
    X, Y = [], []
    for i in range(seq_len, len(vals)):
        X.append(vals[i - seq_len:i].reshape(-1, 1).astype(np.float32))
        Y.append(vals[i] - vals[i - 1])
    return np.array(X), np.array(Y, dtype=np.float32)

def train_model(model, X, Y, epochs=EPOCHS, lr=LR, batch=BATCH):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(X)
    for ep in range(epochs):
        perm = np.random.permutation(n)
        for start in range(0, n, batch):
            idx = perm[start:start + batch]
            x = torch.from_numpy(X[idx]).to(device)
            y = torch.from_numpy(Y[idx]).reshape(-1, 1).to(device)
            opt.zero_grad()
            pred = model(x)
            loss = nn.functional.mse_loss(pred, y)
            loss.backward()
            opt.step()
    return model

class LSTM1Step(nn.Module):
    def __init__(self, in_dim=1, hidden=HIDDEN):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True)
        self.linear = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])

class BiLSTM1Step(nn.Module):
    def __init__(self, hidden=HIDDEN):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True, bidirectional=True)
        self.linear = nn.Linear(2 * hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])

def run_model_standard(model_fn):
    rows = []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + SEQ_LEN + 1:
            continue
        vals = sub[freq].values.astype(np.float32)
        dts = sub["datetime"].values
        train_vals = vals[:-TEST_STEPS]
        test_vals = vals[-TEST_STEPS:]
        test_dts = dts[-TEST_STEPS:]
        X_tr, Y_tr = build_sequences(train_vals, SEQ_LEN)
        if len(X_tr) < BATCH:
            continue
        model = model_fn().to(device)
        train_model(model, X_tr, Y_tr)
        model.eval()
        with torch.no_grad():
            for i in range(TEST_STEPS):
                start = len(train_vals) - SEQ_LEN + i
                end = start + SEQ_LEN
                inp = vals[start:end].reshape(1, SEQ_LEN, 1)
                x = torch.from_numpy(inp).to(device)
                pred = model(x).cpu().numpy().item()
                rows.append({"datetime": test_dts[i], "frequency_band": freq, "actual": float(test_vals[i]), "predicted": float(pred)})
    return pd.DataFrame(rows)

def run_model_residual(model_fn):
    """Train on residual (y_next - y_last); at test time pred = y_last + model(seq)."""
    rows = []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + SEQ_LEN + 1:
            continue
        vals = sub[freq].values.astype(np.float32)
        dts = sub["datetime"].values
        train_vals = vals[:-TEST_STEPS]
        test_vals = vals[-TEST_STEPS:]
        test_dts = dts[-TEST_STEPS:]
        X_tr, Y_tr = build_sequences_residual(train_vals, SEQ_LEN)
        if len(X_tr) < BATCH:
            continue
        model = model_fn().to(device)
        train_model(model, X_tr, Y_tr)
        model.eval()
        with torch.no_grad():
            for i in range(TEST_STEPS):
                start = len(train_vals) - SEQ_LEN + i
                end = start + SEQ_LEN
                last_val = float(vals[end - 1])
                inp = vals[start:end].reshape(1, SEQ_LEN, 1)
                x = torch.from_numpy(inp).to(device)
                delta = model(x).cpu().numpy().item()
                pred = last_val + delta
                rows.append({"datetime": test_dts[i], "frequency_band": freq, "actual": float(test_vals[i]), "predicted": float(pred)})
    return pd.DataFrame(rows)

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

print("Training Residual LSTM and BiLSTM...")
df_residual = run_model_residual(lambda: LSTM1Step())
df_bilstm = run_model_standard(lambda: BiLSTM1Step())

mae_residual = float(np.mean(np.abs(df_residual["actual"] - df_residual["predicted"])))
mae_bilstm = float(np.mean(np.abs(df_bilstm["actual"] - df_bilstm["predicted"])))
mase_residual = mase_from_df(df_residual)
mase_bilstm = mase_from_df(df_bilstm)

# Ensemble: blend naive with best RNN (use residual if it's best, else bilstm)
best_rnn_mae = min(mae_residual, mae_bilstm)
if mae_residual <= mae_bilstm:
    df_rnn = df_residual.rename(columns={"predicted": "pred_rnn"})
else:
    df_rnn = df_bilstm.rename(columns={"predicted": "pred_rnn"})
# Don't include "actual" in df_naive_renamed so merge keeps a single "actual" from df_rnn (avoids actual_x/actual_y)
df_naive_renamed = df_naive.rename(columns={"predicted": "pred_naive"})[["datetime", "frequency_band", "pred_naive"]]
merge_df = df_rnn.merge(df_naive_renamed, on=["datetime", "frequency_band"], how="inner")
# Try a few blend weights
best_mae_ens, best_alpha = np.inf, 0.5
for alpha in [0.3, 0.5, 0.7, 0.9]:
    pred_ens = alpha * merge_df["pred_naive"] + (1 - alpha) * merge_df["pred_rnn"]
    mae_ens = float(np.mean(np.abs(merge_df["actual"] - pred_ens)))
    if mae_ens < best_mae_ens:
        best_mae_ens, best_alpha = mae_ens, alpha
merge_df["predicted"] = best_alpha * merge_df["pred_naive"] + (1 - best_alpha) * merge_df["pred_rnn"]
mase_ens = mase_from_df(merge_df[["datetime", "frequency_band", "actual", "predicted"]])

print("\n=== Extra RNNs vs Naive (1-step-ahead) ===")
print(f"  Naive:          MAE = {mae_naive:.4f},  MASE = {mase_naive:.4f}" if not np.isnan(mase_naive) else f"  Naive:          MAE = {mae_naive:.4f},  MASE = —")
print(f"  Residual LSTM:  MAE = {mae_residual:.4f},  MASE = {mase_residual:.4f}" if not np.isnan(mase_residual) else f"  Residual LSTM:  MAE = {mae_residual:.4f},  MASE = —")
print(f"  BiLSTM:        MAE = {mae_bilstm:.4f},  MASE = {mase_bilstm:.4f}" if not np.isnan(mase_bilstm) else f"  BiLSTM:        MAE = {mae_bilstm:.4f},  MASE = —")
print(f"  Ensemble (α={best_alpha:.1f}): MAE = {best_mae_ens:.4f},  MASE = {mase_ens:.4f}" if not np.isnan(mase_ens) else f"  Ensemble (α={best_alpha:.1f}): MAE = {best_mae_ens:.4f},  MASE = —")

all_maes = {"Naive": mae_naive, "Residual LSTM": mae_residual, "BiLSTM": mae_bilstm, "Ensemble": best_mae_ens}
winner = min(all_maes, key=all_maes.get)
print(f"\n  → Best MAE: {all_maes[winner]:.4f} ({winner})")
if best_mae_ens < mae_naive or mae_residual < mae_naive or mae_bilstm < mae_naive:
    print("  → At least one extra RNN or ensemble beats naive at 1-step.")
else:
    print("  → Naive still wins (typical for very persistent 1-step series).")
ALL_RESULTS.append({"model": "Residual LSTM", "mae": mae_residual, "mase": mase_residual})
ALL_RESULTS.append({"model": "BiLSTM", "mae": mae_bilstm, "mase": mase_bilstm})
ALL_RESULTS.append({"model": f"Ensemble (α={best_alpha:.1f})", "mae": best_mae_ens, "mase": mase_ens})

Training Residual LSTM and BiLSTM...


KeyError: 'actual'

In [12]:
import json
import torch
import torch.nn as nn

SEQ_LEN = 24   # lookback hours (one day)
HIDDEN = 32
EPOCHS = 15
BATCH = 64
LR = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LSTM1Step(nn.Module):
    def __init__(self, hidden=HIDDEN):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.linear = nn.Linear(hidden, 1)
    def forward(self, x):
        # x: (batch, seq_len, 1)
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])  # (batch, 1)

def build_sequences(vals, seq_len):
    X, Y = [], []
    for i in range(seq_len, len(vals)):
        X.append(vals[i - seq_len:i].reshape(-1, 1))
        Y.append(vals[i])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

def train_lstm(model, X, Y, epochs=EPOCHS, lr=LR, batch=BATCH):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = len(X)
    for ep in range(epochs):
        perm = np.random.permutation(n)
        for start in range(0, n, batch):
            idx = perm[start:start + batch]
            x = torch.from_numpy(X[idx]).to(device)
            y = torch.from_numpy(Y[idx]).reshape(-1, 1).to(device)
            opt.zero_grad()
            pred = model(x)
            loss = nn.functional.mse_loss(pred, y)
            loss.backward()
            opt.step()
    return model

rows_lstm = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + SEQ_LEN + 1:
        continue
    vals = sub[freq].values.astype(np.float32)
    dts = sub["datetime"].values
    train_vals = vals[: -(TEST_STEPS)]
    test_vals = vals[-TEST_STEPS:]
    test_dts = dts[-TEST_STEPS:]
    X_tr, Y_tr = build_sequences(train_vals, SEQ_LEN)
    if len(X_tr) < BATCH:
        continue
    model = LSTM1Step().to(device)
    train_lstm(model, X_tr, Y_tr)
    model.eval()
    with torch.no_grad():
        for i in range(TEST_STEPS):
            start = len(train_vals) - SEQ_LEN + i
            end = start + SEQ_LEN
            inp = vals[start:end].reshape(1, SEQ_LEN, 1)
            x = torch.from_numpy(inp).to(device)
            pred = model(x).cpu().numpy().item()
            rows_lstm.append({"datetime": test_dts[i], "frequency_band": freq, "actual": float(test_vals[i]), "predicted": float(pred)})

pred_df_lstm = pd.DataFrame(rows_lstm)
pred_df_lstm["datetime"] = pd.to_datetime(pred_df_lstm["datetime"])
if len(pred_df_lstm) == 0:
    raise ValueError("No LSTM predictions; need enough data per band (TEST_STEPS + SEQ_LEN + 1).")

print("LSTM 1-step-ahead (sample):")
print(pred_df_lstm.head(10).to_string(index=False))
print("...")
print(pred_df_lstm.tail(5).to_string(index=False))

y_true = pred_df_lstm["actual"].values
y_pred = pred_df_lstm["predicted"].values
mae_lstm = float(np.mean(np.abs(y_true - y_pred)))
rmse_lstm = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
_mase_diffs = pred_df_lstm.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(_mase_diffs.mean()) if len(_mase_diffs) > 0 else np.nan
mase_lstm = float(mae_lstm / mase_denom) if mase_denom and mase_denom > 0 else np.nan

print(f"\nMAE (LSTM):  {mae_lstm:.4f}")
print(f"RMSE (LSTM): {rmse_lstm:.4f}")
print(f"MASE (LSTM): {mase_lstm:.4f}" if not np.isnan(mase_lstm) else "MASE (LSTM): —")

out_dir = _root / "data" / "time_series"
out_dir.mkdir(parents=True, exist_ok=True)
metrics_lstm = {"mae": mae_lstm, "rmse": rmse_lstm, "mase": mase_lstm, "n_predictions": len(pred_df_lstm), "test_steps_hours": TEST_STEPS, "seq_len": SEQ_LEN, "hidden": HIDDEN, "epochs": EPOCHS}
with open(out_dir / "lstm_metrics.json", "w") as f:
    json.dump(metrics_lstm, f, indent=2)
pred_df_lstm.to_parquet(out_dir / "lstm_predictions.parquet", index=False)
print(f"\nStored: {out_dir / 'lstm_metrics.json'}, {out_dir / 'lstm_predictions.parquet'}")

LSTM 1-step-ahead (sample):
           datetime frequency_band    actual  predicted
2025-01-19 00:00:00    3.100-3.105 15.202703   4.652444
2025-01-19 01:00:00    3.100-3.105 12.972973   4.687499
2025-01-19 02:00:00    3.100-3.105 10.337838   4.699624
2025-01-19 03:00:00    3.100-3.105  9.932432   4.704426
2025-01-19 04:00:00    3.100-3.105  8.716216   4.706497
2025-01-19 05:00:00    3.100-3.105 13.581081   4.706493
2025-01-19 06:00:00    3.100-3.105 16.013514   4.706871
2025-01-19 07:00:00    3.100-3.105 27.770269   4.702858
2025-01-19 08:00:00    3.100-3.105 28.581081   4.674225
2025-01-19 09:00:00    3.100-3.105 42.972973   4.666504
...
           datetime frequency_band    actual  predicted
2026-01-18 19:00:00    3.445-3.450 78.738739   4.828372
2026-01-18 20:00:00    3.445-3.450 57.117119   4.803891
2026-01-18 21:00:00    3.445-3.450 79.099098   4.958981
2026-01-18 22:00:00    3.445-3.450 79.099098   4.836436
2026-01-18 23:00:00    3.445-3.450 78.198196   4.764722

MAE (LSTM):  11

## Experiment lab: seasonality-aware models (beat naive)

Because **naive** (last value) keeps winning at 1-step despite visible seasonality, we try models that *explicitly* use seasonal lags and linear combinations:

- **Ridge on lags**: Features = [y(t-1), y(t-24), y(t-168)] — last hour, same hour yesterday, same hour last week. Ridge regression per band.
- **Learned blend (naive + seasonal)**: For 1-step, pred = α·naive + (1−α)·seasonal; choose α per band or globally to minimize MAE on train/validation.
- **LSTM lookback 168**: Same LSTM but SEQ_LEN=168 (one week) so the RNN sees weekly cycle.
- **Seasonal residual**: Predict *delta from seasonal naive*: target = y(t+1) − y(t−23); forecast = y(t−23) + predicted delta.

Below we run these and print a **summary table** of all methods (baselines + these) so you can see if any non-naive model wins.

In [ ]:
# Experiment lab: Ridge on lags, learned blend, LSTM-168, seasonal residual. Summary table at end.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import Ridge

# --- Config (same as above) ---
SEQ_LEN = 24
SEQ_LEN_WEEK = 168
HIDDEN = 32
EPOCHS = 25
BATCH = 64
LR = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = []  # list of {"model": name, "mae": float, "mase": float}

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

# --- 1) Naive baseline ---
rows_naive = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 1:
        continue
    vals, dts = sub[freq].values.astype(np.float64), sub["datetime"].values
    test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
df_naive = pd.DataFrame(rows_naive)
mae_naive = float(np.mean(np.abs(df_naive["actual"] - df_naive["predicted"])))
mase_naive = mase_from_df(df_naive)
results.append({"model": "Naive", "mae": mae_naive, "mase": mase_naive})

# --- 2) Seasonal naive (1-step: same hour yesterday) ---
rows_seasonal = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 24:
        continue
    vals, dts = sub[freq].values.astype(np.float64), sub["datetime"].values
    test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    for i in range(23, TEST_STEPS - 1):
        rows_seasonal.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i-23]})
df_seasonal = pd.DataFrame(rows_seasonal)
mae_seasonal = float(np.mean(np.abs(df_seasonal["actual"] - df_seasonal["predicted"])))
mase_seasonal = mase_from_df(df_seasonal)
results.append({"model": "Seasonal naive", "mae": mae_seasonal, "mase": mase_seasonal})

# --- 3) Ridge on [y(t-1), y(t-24), y(t-168)] ---
LAG24, LAG168 = 24, 168
rows_ridge = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + LAG168 + 1:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    train_vals, test_vals, test_dts = vals[:-TEST_STEPS], vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    T = len(train_vals)
    X_tr, Y_tr = [], []
    for i in range(LAG168, len(train_vals) - 1):
        X_tr.append([train_vals[i-1], train_vals[i-LAG24], train_vals[i-LAG168]])
        Y_tr.append(train_vals[i+1])
    X_tr, Y_tr = np.array(X_tr), np.array(Y_tr)
    if len(X_tr) < 10:
        continue
    ridge = Ridge(alpha=1.0).fit(X_tr, Y_tr)
    for i in range(TEST_STEPS - 1):
        if T + i - LAG168 < 0:
            continue
        feat = np.array([[vals[T+i-1], vals[T+i-LAG24], vals[T+i-LAG168]]])
        pred = ridge.predict(feat).item()
        rows_ridge.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": pred})
df_ridge = pd.DataFrame(rows_ridge)
if len(df_ridge) > 0:
    mae_ridge = float(np.mean(np.abs(df_ridge["actual"] - df_ridge["predicted"])))
    mase_ridge = mase_from_df(df_ridge)
    results.append({"model": "Ridge (lag1,24,168)", "mae": mae_ridge, "mase": mase_ridge})

# --- 4) Learned blend: alpha*naive + (1-alpha)*seasonal (per-band alpha on train) ---
df_naive_ = df_naive.rename(columns={"predicted": "p_naive"})[["datetime", "frequency_band", "actual", "p_naive"]]
df_seasonal_ = df_seasonal.rename(columns={"predicted": "p_seasonal"})[["datetime", "frequency_band", "p_seasonal"]]
merge_b = df_naive_.merge(df_seasonal_, on=["datetime", "frequency_band"], how="inner")
best_alpha_global, best_mae_b = 0.5, np.inf
for alpha in np.linspace(0, 1, 21):
    pred = alpha * merge_b["p_naive"] + (1 - alpha) * merge_b["p_seasonal"]
    mae = float(np.mean(np.abs(merge_b["actual"] - pred)))
    if mae < best_mae_b:
        best_mae_b, best_alpha_global = mae, alpha
merge_b["predicted"] = best_alpha_global * merge_b["p_naive"] + (1 - best_alpha_global) * merge_b["p_seasonal"]
mase_blend = mase_from_df(merge_b[["datetime", "frequency_band", "actual", "predicted"]])
results.append({"model": f"Blend naive+seasonal (α={best_alpha_global:.2f})", "mae": best_mae_b, "mase": mase_blend})

# --- 5) LSTM with 168-step lookback ---
def build_seq(vals, L):
    X, Y = [], []
    for i in range(L, len(vals)):
        X.append(vals[i-L:i].reshape(-1, 1).astype(np.float32))
        Y.append(vals[i])
    return np.array(X), np.array(Y, dtype=np.float32)

class LSTM1Step(nn.Module):
    def __init__(self, in_dim=1, hidden=HIDDEN):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, batch_first=True)
        self.linear = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])

def train_run_lstm(seq_len):
    rows = []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + seq_len + 1:
            continue
        vals = sub[freq].values.astype(np.float32)
        dts = sub["datetime"].values
        train_vals, test_vals, test_dts = vals[:-TEST_STEPS], vals[-TEST_STEPS:], dts[-TEST_STEPS:]
        X_tr, Y_tr = build_seq(train_vals, seq_len)
        if len(X_tr) < BATCH:
            continue
        model = LSTM1Step().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        for _ in range(EPOCHS):
            perm = np.random.permutation(len(X_tr))
            for start in range(0, len(X_tr), BATCH):
                idx = perm[start:start+BATCH]
                x = torch.from_numpy(X_tr[idx]).to(device)
                y = torch.from_numpy(Y_tr[idx]).reshape(-1, 1).to(device)
                opt.zero_grad()
                loss = nn.functional.mse_loss(model(x), y)
                loss.backward()
                opt.step()
        model.eval()
        with torch.no_grad():
            for i in range(TEST_STEPS):
                start = len(train_vals) - seq_len + i
                inp = vals[start:start+seq_len].reshape(1, seq_len, 1)
                pred = model(torch.from_numpy(inp).to(device)).cpu().numpy().item()
                rows.append({"datetime": test_dts[i], "frequency_band": freq, "actual": float(test_vals[i]), "predicted": float(pred)})
    return pd.DataFrame(rows)

print("Training LSTM-168 (one week lookback)...")
df_lstm168 = train_run_lstm(SEQ_LEN_WEEK)
mae_lstm168 = float(np.mean(np.abs(df_lstm168["actual"] - df_lstm168["predicted"])))
mase_lstm168 = mase_from_df(df_lstm168)
results.append({"model": "LSTM (seq=168)", "mae": mae_lstm168, "mase": mase_lstm168})

# --- 6) Seasonal residual: predict y(t+1) - y(t-23), then pred = y(t-23) + delta ---
def build_seasonal_residual(vals, seq_len):
    X, Y = [], []
    for i in range(24, len(vals) - 1):
        if i - seq_len < 0:
            continue
        X.append(vals[i - seq_len : i].reshape(-1, 1).astype(np.float32))
        Y.append(vals[i + 1] - vals[i - 23])
    return np.array(X), np.array(Y, dtype=np.float32)

rows_sres = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + SEQ_LEN + 24:
        continue
    vals = sub[freq].values.astype(np.float32)
    dts = sub["datetime"].values
    train_vals, test_vals, test_dts = vals[:-TEST_STEPS], vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    X_tr, Y_tr = build_seasonal_residual(train_vals, SEQ_LEN)
    if len(X_tr) < BATCH:
        continue
    model = LSTM1Step().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for _ in range(EPOCHS):
        perm = np.random.permutation(len(X_tr))
        for start in range(0, len(X_tr), BATCH):
            idx = perm[start:start+BATCH]
            x = torch.from_numpy(X_tr[idx]).to(device)
            y = torch.from_numpy(Y_tr[idx]).reshape(-1, 1).to(device)
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(x), y)
            loss.backward()
            opt.step()
    model.eval()
    T = len(train_vals)
    with torch.no_grad():
        for i in range(23, TEST_STEPS - 1):
            # Input: 24 values ending at T+i (last known). Seasonal ref for y(T+i+1) = y(T+i-23).
            start = T + i - 23
            if start + SEQ_LEN > len(vals):
                continue
            inp = vals[start:start+SEQ_LEN].reshape(1, SEQ_LEN, 1)
            delta = model(torch.from_numpy(inp).to(device)).cpu().numpy().item()
            pred = float(vals[T + i - 23]) + delta
            rows_sres.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": float(test_vals[i+1]), "predicted": pred})
df_sres = pd.DataFrame(rows_sres)
if len(df_sres) > 0:
    mae_sres = float(np.mean(np.abs(df_sres["actual"] - df_sres["predicted"])))
    mase_sres = mase_from_df(df_sres)
    results.append({"model": "Seasonal residual LSTM", "mae": mae_sres, "mase": mase_sres})

# --- Summary table ---
print("\n" + "="*60)
print("SUMMARY: All models (1-step-ahead) — sorted by MAE")
print("="*60)
results_sorted = sorted(results, key=lambda x: x["mae"])
for r in results_sorted:
    mase_str = f"{r['mase']:.4f}" if not np.isnan(r['mase']) else "—"
    print(f"  {r['model']:<35}  MAE = {r['mae']:.4f}   MASE = {mase_str}")
best = results_sorted[0]
print("="*60)
print(f"  → Best: {best['model']} (MAE = {best['mae']:.4f})")
if best["model"] != "Naive":
    print("  → A non-naive model wins.")
else:
    print("  → Naive still best; try more architectures or longer horizons.")
for r in results:
    if r["model"] not in ("Naive", "Seasonal naive"):
        ALL_RESULTS.append(r)

Training LSTM-168 (one week lookback)...


## Model search: ElasticNet, Random Forest, Gradient Boosting, MLP (beat naive)

Classical ML on lag features [y(t-1), y(t-24), y(t-168)]. Same 1-step-ahead test set and MAE/MASE. Results are appended to `ALL_RESULTS` for the final summary.

In [ ]:
# Classical ML on lags [y(t-1), y(t-24), y(t-168)]. Requires: df, freq_bands, TEST_STEPS, ALL_RESULTS from earlier cells.
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

LAG24, LAG168 = 24, 168

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

def run_lag_model(name, model_factory):
    rows = []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + LAG168 + 1:
            continue
        vals = sub[freq].values.astype(np.float64)
        dts = sub["datetime"].values
        train_vals, test_vals, test_dts = vals[:-TEST_STEPS], vals[-TEST_STEPS:], dts[-TEST_STEPS:]
        T = len(train_vals)
        X_tr, Y_tr = [], []
        for i in range(LAG168, len(train_vals) - 1):
            X_tr.append([train_vals[i-1], train_vals[i-LAG24], train_vals[i-LAG168]])
            Y_tr.append(train_vals[i+1])
        X_tr, Y_tr = np.array(X_tr), np.array(Y_tr)
        if len(X_tr) < 10:
            continue
        model = model_factory()
        model.fit(X_tr, Y_tr)
        for i in range(TEST_STEPS - 1):
            if T + i - LAG168 < 0:
                continue
            feat = np.array([[vals[T+i-1], vals[T+i-LAG24], vals[T+i-LAG168]]])
            pred = model.predict(feat).item()
            rows.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": pred})
    return pd.DataFrame(rows)

print("Training ElasticNet, Random Forest, Gradient Boosting, MLP on lags...")
df_elastic = run_lag_model("ElasticNet", lambda: ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=2000))
df_rf = run_lag_model("Random Forest", lambda: RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42))
df_gbm = run_lag_model("Gradient Boosting", lambda: GradientBoostingRegressor(n_estimators=50, max_depth=4, learning_rate=0.1, random_state=42))
df_mlp = run_lag_model("MLP", lambda: MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=200, random_state=42))

for name, d in [("ElasticNet (lags)", df_elastic), ("Random Forest (lags)", df_rf), ("Gradient Boosting (lags)", df_gbm), ("MLP (lags)", df_mlp)]:
    if len(d) > 0:
        mae = float(np.mean(np.abs(d["actual"] - d["predicted"])))
        mase = mase_from_df(d)
        ALL_RESULTS.append({"model": name, "mae": mae, "mase": mase})
        print(f"  {name}: MAE = {mae:.4f}, MASE = {mase:.4f}" if not np.isnan(mase) else f"  {name}: MAE = {mae:.4f}")
print("Done. Run the Final summary cell below to see all models sorted by MAE.")

## More ML models on lags

Additional models on the same lag features [y(t-1), y(t-24), y(t-168)]. Any that come close to (or beat) naive are added to `ALL_RESULTS` for the final summary.

In [17]:
# More ML models on [y(t-1), y(t-24), y(t-168)]. Requires: df, freq_bands, TEST_STEPS, ALL_RESULTS.
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, BayesianRidge, LinearRegression, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

LAG24, LAG168 = 24, 168
if "ALL_RESULTS" not in globals() or not isinstance(ALL_RESULTS, list):
    ALL_RESULTS = []

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

def run_lag_model(model_factory):
    rows = []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + LAG168 + 1:
            continue
        vals = sub[freq].values.astype(np.float64)
        dts = sub["datetime"].values
        train_vals, test_vals, test_dts = vals[:-TEST_STEPS], vals[-TEST_STEPS:], dts[-TEST_STEPS:]
        T = len(train_vals)
        X_tr, Y_tr = [], []
        for i in range(LAG168, len(train_vals) - 1):
            X_tr.append([train_vals[i-1], train_vals[i-LAG24], train_vals[i-LAG168]])
            Y_tr.append(train_vals[i+1])
        X_tr, Y_tr = np.array(X_tr), np.array(Y_tr)
        if len(X_tr) < 10:
            continue
        model = model_factory()
        model.fit(X_tr, Y_tr)
        for i in range(TEST_STEPS - 1):
            if T + i - LAG168 < 0:
                continue
            feat = np.array([[vals[T+i-1], vals[T+i-LAG24], vals[T+i-LAG168]]])
            pred = model.predict(feat).item()
            rows.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": pred})
    return pd.DataFrame(rows)

def add_model(name, model_factory, naive_mae=2.42):
    try:
        d = run_lag_model(model_factory)
        if len(d) == 0:
            return
        mae = float(np.mean(np.abs(d["actual"] - d["predicted"])))
        mase = mase_from_df(d)
        ALL_RESULTS.append({"model": name, "mae": mae, "mase": mase})
        close = " (close!)" if mae < naive_mae * 1.05 else ""
        print(f"  {name}: MAE = {mae:.4f}, MASE = {mase:.4f}{close}" if not np.isnan(mase) else f"  {name}: MAE = {mae:.4f}{close}")
    except Exception as e:
        print(f"  {name}: skip ({e})")

print("Training more ML models on lags...")
add_model("Lasso (lags)", lambda: Lasso(alpha=0.01, max_iter=2000))
add_model("BayesianRidge (lags)", lambda: BayesianRidge())
add_model("OLS / LinearRegression (lags)", lambda: LinearRegression())
add_model("HuberRegressor (lags)", lambda: HuberRegressor(epsilon=1.2))
add_model("KNN k=5 (lags)", lambda: KNeighborsRegressor(n_neighbors=5))
add_model("KNN k=10 (lags)", lambda: KNeighborsRegressor(n_neighbors=10))
add_model("ExtraTrees (lags)", lambda: ExtraTreesRegressor(n_estimators=30, max_depth=8, random_state=42))
add_model("HistGradientBoosting (lags)", lambda: HistGradientBoostingRegressor(max_iter=50, max_depth=5, random_state=42))
add_model("MLP (64,32,16)", lambda: MLPRegressor(hidden_layer_sizes=(64, 32, 16), max_iter=300, random_state=42))
add_model("MLP (16,8) early", lambda: MLPRegressor(hidden_layer_sizes=(16, 8), max_iter=150, random_state=42))
print("Done. Run Final summary to see all models.")

Training more ML models on lags...
  Lasso (lags): MAE = 4.6509, MASE = 1.9218
  BayesianRidge (lags): MAE = 4.6705, MASE = 1.9299
  OLS / LinearRegression (lags): MAE = 4.6520, MASE = 1.9223


/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_

  HuberRegressor (lags): MAE = 4.2174, MASE = 1.7427
  KNN k=5 (lags): MAE = 6.3082, MASE = 2.6066
  KNN k=10 (lags): MAE = 6.2514, MASE = 2.5832
  ExtraTrees (lags): MAE = 6.3726, MASE = 2.6333
  HistGradientBoosting (lags): MAE = 6.3830, MASE = 2.6376


/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) 

  MLP (64,32,16): MAE = 4.2727, MASE = 1.7655


/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/pradeep/workspace/nikhita/cablelabs/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) 

  MLP (16,8) early: MAE = 4.0470, MASE = 1.6723
Done. Run Final summary to see all models.


## Background experiment winner: Tuned blend (α=0.90)

A background script (`tmp_beat_naive_mini.py`) ran Naive, Ridge on lags, and a **tuned blend** α·naive + (1−α)·seasonal. The blend with **α=0.90** beat the naive baseline (MAE 2.39 vs 2.42). Below we add this model to the notebook and register it in `ALL_RESULTS`.

In [14]:
# Background experiment winner: Tuned blend α=0.90 (0.9*naive + 0.1*seasonal). Requires: df, freq_bands, TEST_STEPS.
import numpy as np
import pandas as pd

if "ALL_RESULTS" not in globals() or not isinstance(ALL_RESULTS, list):
    ALL_RESULTS = []  # create if not yet defined (run data-load cell first for full pipeline)

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

# 1-step naive and seasonal naive (same hour yesterday)
rows_naive, rows_seasonal = [], []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 24:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
    for i in range(23, TEST_STEPS - 1):
        rows_seasonal.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i-23]})
df_naive_b = pd.DataFrame(rows_naive)
df_seasonal_b = pd.DataFrame(rows_seasonal)
df_naive_b = df_naive_b.rename(columns={"predicted": "p_naive"})[["datetime", "frequency_band", "actual", "p_naive"]]
df_seasonal_b = df_seasonal_b.rename(columns={"predicted": "p_seasonal"})[["datetime", "frequency_band", "p_seasonal"]]
merge_b = df_naive_b.merge(df_seasonal_b, on=["datetime", "frequency_band"], how="inner")

# Winning α from background experiment
ALPHA_WIN = 0.90
merge_b["predicted"] = ALPHA_WIN * merge_b["p_naive"] + (1 - ALPHA_WIN) * merge_b["p_seasonal"]
mae_blend = float(np.mean(np.abs(merge_b["actual"] - merge_b["predicted"])))
mase_blend = mase_from_df(merge_b[["datetime", "frequency_band", "actual", "predicted"]])
ALL_RESULTS.append({"model": f"Tuned blend (α={ALPHA_WIN}) [experiment winner]", "mae": mae_blend, "mase": mase_blend})
print(f"Tuned blend (α={ALPHA_WIN}): MAE = {mae_blend:.4f}, MASE = {mase_blend:.4f}")
print("(Added to ALL_RESULTS. Run Final summary cell to see full table.)")

NameError: name 'ALL_RESULTS' is not defined

## Alpha sweep: find best blend α

Sweep α from 0 to 1 (pred = α·naive + (1−α)·seasonal) and report MAE for each. Register the **best α** in `ALL_RESULTS` so we can see if we beat the fixed α=0.90.

In [15]:
# Alpha sweep: pred = α*naive + (1-α)*seasonal. Requires: df, freq_bands, TEST_STEPS; uses merge_b from Tuned blend cell if available.
import numpy as np
import pandas as pd

def mase_from_df(df):
    diffs = df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(df["actual"] - df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

# Build naive + seasonal merge if not already in scope (same logic as Tuned blend cell)
if "merge_b" not in globals() or "p_naive" not in (merge_b.columns if hasattr(merge_b, "columns") else []):
    rows_naive, rows_seasonal = [], []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + 24:
            continue
        vals = sub[freq].values.astype(np.float64)
        dts = sub["datetime"].values
        test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
        for i in range(0, TEST_STEPS - 1):
            rows_naive.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
        for i in range(23, TEST_STEPS - 1):
            rows_seasonal.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i-23]})
    df_naive_b = pd.DataFrame(rows_naive).rename(columns={"predicted": "p_naive"})[["datetime", "frequency_band", "actual", "p_naive"]]
    df_seasonal_b = pd.DataFrame(rows_seasonal).rename(columns={"predicted": "p_seasonal"})[["datetime", "frequency_band", "p_seasonal"]]
    merge_b = df_naive_b.merge(df_seasonal_b, on=["datetime", "frequency_band"], how="inner")

# Sweep alpha (finer around 0.85–0.95 where we expect the best)
alphas = np.sort(np.unique(np.concatenate([np.linspace(0, 1, 51), np.linspace(0.85, 0.95, 21)])))
sweep = []
for alpha in alphas:
    pred = alpha * merge_b["p_naive"] + (1 - alpha) * merge_b["p_seasonal"]
    mae = float(np.mean(np.abs(merge_b["actual"] - pred)))
    sweep.append({"alpha": alpha, "mae": mae})

sweep_df = pd.DataFrame(sweep)
best_row = sweep_df.loc[sweep_df["mae"].idxmin()]
best_alpha = float(best_row["alpha"])
best_mae = float(best_row["mae"])

# MASE for best blend
merge_b["predicted"] = best_alpha * merge_b["p_naive"] + (1 - best_alpha) * merge_b["p_seasonal"]
mase_best = mase_from_df(merge_b[["datetime", "frequency_band", "actual", "predicted"]])

if "ALL_RESULTS" not in globals() or not isinstance(ALL_RESULTS, list):
    ALL_RESULTS = []
ALL_RESULTS.append({"model": f"Blend best α={best_alpha:.3f}", "mae": best_mae, "mase": mase_best})

# Show sweep (sample)
print("Alpha sweep (sample):")
print(sweep_df.iloc[:: max(1, len(sweep_df) // 15)].to_string(index=False))
print(f"\nBest: α = {best_alpha:.3f}  →  MAE = {best_mae:.4f},  MASE = {mase_best:.4f}")
print(f"(Added 'Blend best α={best_alpha:.3f}' to ALL_RESULTS.)")

Alpha sweep (sample):
 alpha      mae
 0.000 4.290000
 0.080 4.005791
 0.160 3.738977
 0.240 3.491405
 0.320 3.264162
 0.400 3.058735
 0.480 2.877667
 0.560 2.721856
 0.640 2.592566
 0.720 2.492373
 0.800 2.425188
 0.855 2.398820
 0.875 2.393460
 0.895 2.390558
 0.910 2.389979
 0.925 2.390851
 0.940 2.393258
 0.980 2.406924

Best: α = 0.910  →  MAE = 2.3900,  MASE = 0.9884
(Added 'Blend best α=0.910' to ALL_RESULTS.)


## Final summary (all models tried)

Run **all model cells above** (data load → naive vs seasonal → GRU/Stacked/LSTM+lag24 → Residual/BiLSTM/Ensemble → Experiment lab → Model search), then run the cell below to see the full table of every model sorted by MAE and a short conclusion.

In [18]:
# Final summary: all models (1-step-ahead), sorted by MAE. Requires ALL_RESULTS from running the model cells above.
import numpy as np

if not ALL_RESULTS:
    print("ALL_RESULTS is empty. Run the data-load cell and the model cells (naive vs seasonal, GRU/Stacked/LSTM+lag24, Residual/BiLSTM, Experiment lab, Model search) first.")
else:
    sorted_results = sorted(ALL_RESULTS, key=lambda x: x["mae"])
    print("=" * 65)
    print("FINAL SUMMARY: All models (1-step-ahead) — sorted by MAE")
    print("=" * 65)
    for r in sorted_results:
        mase_str = f"{r['mase']:.4f}" if not np.isnan(r['mase']) else "—"
        print(f"  {r['model']:<38}  MAE = {r['mae']:.4f}   MASE = {mase_str}")
    print("=" * 65)
    best = sorted_results[0]
    print(f"  → Best model: {best['model']} (MAE = {best['mae']:.4f})")
    naive_mae = next((x["mae"] for x in ALL_RESULTS if x["model"] == "Naive"), None)
    if naive_mae is not None and best["mae"] < naive_mae - 1e-6:
        print("  → A non-naive model beats the naive baseline.")
    else:
        print("  → Naive remained best (or tied). Next steps: multi-step evaluation, different horizon, or more feature engineering.")

FINAL SUMMARY: All models (1-step-ahead) — sorted by MAE
  Blend best α=0.910                      MAE = 2.3900   MASE = 0.9884
  MLP (16,8) early                        MAE = 4.0470   MASE = 1.6723
  HuberRegressor (lags)                   MAE = 4.2174   MASE = 1.7427
  MLP (64,32,16)                          MAE = 4.2727   MASE = 1.7655
  Lasso (lags)                            MAE = 4.6509   MASE = 1.9218
  OLS / LinearRegression (lags)           MAE = 4.6520   MASE = 1.9223
  BayesianRidge (lags)                    MAE = 4.6705   MASE = 1.9299
  KNN k=10 (lags)                         MAE = 6.2514   MASE = 2.5832
  KNN k=5 (lags)                          MAE = 6.3082   MASE = 2.6066
  ExtraTrees (lags)                       MAE = 6.3726   MASE = 2.6333
  HistGradientBoosting (lags)             MAE = 6.3830   MASE = 2.6376
  → Best model: Blend best α=0.910 (MAE = 2.3900)
  → Naive remained best (or tied). Next steps: multi-step evaluation, different horizon, or more feature enginee